# Вычислительная ниткография

Алгоритм по портретной фотографии строт схему расположения нитей.
Метод основан на **Filtered Back Projection** из рентгеновской томографии.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from string_art import (
    load_grayscale_square,
    prepare_fbp_image,
    radon_transform,
    filter_sinogram,
    keep_brightest_per_angle,
    normalize_sinogram_probabilities,
    binary_dither,
    binary_sinogram_to_lines,
    draw_threads,
    circle_mask,
)

plt.rcParams['figure.figsize'] = (6, 6)
plt.rcParams['axes.titlesize'] = 13

## Исходное изображение

Загружаем, проверяем квадратную и одноканальную картинку.

In [ ]:
image = load_grayscale_square('input.png')
print(f'Размер: {image.shape[0]}×{image.shape[1]}, диапазон: [{image.min():.2f}, {image.max():.2f}]')

plt.imshow(image, cmap='gray', vmin=0, vmax=1)
plt.title('Исходное фото')
plt.axis('off')
plt.show()

## Подготовка

Поднятием общей яркости исходного изображения устраняем отрицательные значения после рамп-фильтра.
Инвертируем, чтобы тёмные детали (глаза, волосы) получили больше нитей.

In [ ]:
target = prepare_fbp_image(image, brightness_lift=0.2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('До подготовки')
axes[0].axis('off')
axes[1].imshow(target, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('После (инверсия + подъём яркости)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Шаг 1. Преобразование Радона

> *«$R[f(x,y)](\rho,\varphi) = \int_{-\infty}^{\infty} f(\rho\cos\varphi - l\sin\varphi,\; \rho\sin\varphi + l\cos\varphi)\,dl$»*

Для каждого угла $\varphi$ проводим набор параллельных прямых и суммируем яркость вдоль каждой.
Результат — **синограмма**: строки = углы, столбцы = смещения $\rho$.

In [ ]:
ANGLES = 360

sinogram, angles = radon_transform(target, ANGLES)
print(f'Синограмма: {sinogram.shape[0]} углов × {sinogram.shape[1]} смещений')

plt.figure(figsize=(8, 4))
plt.imshow(sinogram, cmap='gray', aspect='auto',
           extent=[0, sinogram.shape[1], angles[-1], 0])
plt.xlabel('Смещение ρ (пиксели)')
plt.ylabel('Угол φ (градусы)')
plt.title('Шаг 1: Синограмма — преобразование Радона')
plt.colorbar(label='Сумма яркости')
plt.tight_layout()
plt.show()

## Шаг 2. Свертка с рамп-фильтром

> *«$F_r(\rho)$ при $\rho=0$: $1/4$; при чётном $\rho$: $0$; при нечётном: $-1/(\pi\rho)^2$»*

Каждую проекцию сворачиваем с этим фильтром — подчёркиваем границы и детали.
После фильтрации могут появиться отрицательные значения.

In [ ]:
filtered = filter_sinogram(sinogram)
print(f'Минимум после фильтрации: {filtered.min():.2f} (отрицательные — нормально)')

plt.figure(figsize=(8, 4))
plt.imshow(filtered, cmap='RdBu', aspect='auto',
           extent=[0, filtered.shape[1], angles[-1], 0],
           vmin=-np.abs(filtered).max(), vmax=np.abs(filtered).max())
plt.xlabel('Смещение ρ (пиксели)')
plt.ylabel('Угол φ (градусы)')
plt.title('Шаг 2: После рамп-фильтра (красный — отрицательные)')
plt.colorbar(label='Значение')
plt.tight_layout()
plt.show()

## Шаг 3. Прореживание

Прореживание проекций по пространственной переменной до заданного числа нитей.

Из всех возможных прямых под каждым углом оставляем только **40 самых ярких**.
Остальные зануляем — нитей будет регулируемое количество.

In [ ]:
THREADS = 40

thinned = keep_brightest_per_angle(filtered, THREADS)
nonzero = (thinned > 0).sum()
total = thinned.size
print(f'Оставлено {nonzero} из {total} ячеек ({100*nonzero/total:.1f}%)')

plt.figure(figsize=(8, 4))
plt.imshow(thinned != 0, cmap='gray', aspect='auto',
           extent=[0, thinned.shape[1], angles[-1], 0])
plt.xlabel('Смещение ρ (пиксели)')
plt.ylabel('Угол φ (градусы)')
plt.title(f'Шаг 3: После прореживания (не более {THREADS} на угол)')
plt.tight_layout()
plt.show()

## Шаг 4. Нормировка

Занулим точки синограммы, значения которых меньше 0; нормируем оставшийся диапазон к отрезку $[0,1]$

Отрицательные → 0, остальные приводим к вероятностям от 0 до 1.
Теперь каждое число = вероятность того, что нить будет протянута.

In [ ]:
probabilities = normalize_sinogram_probabilities(thinned, gamma=0.85)
print(f'Диапазон вероятностей: [{probabilities.min():.3f}, {probabilities.max():.3f}]')

plt.figure(figsize=(8, 4))
plt.imshow(probabilities, cmap='hot', aspect='auto',
           extent=[0, probabilities.shape[1], angles[-1], 0],
           vmin=0, vmax=1)
plt.xlabel('Смещение ρ (пиксели)')
plt.ylabel('Угол φ (градусы)')
plt.title('Шаг 4: Вероятности (нормировка к [0, 1])')
plt.colorbar(label='Вероятность нити')
plt.tight_layout()
plt.show()

## Шаг 5. Бинарное распыление

В каждом пикселе выходной синограммы поставим 1 с вероятностью, указанной в этом пикселе, иначе поставим 0

In [ ]:
binary = binary_dither(probabilities, seed=1)
thread_count = binary.sum()
print(f'Нитей: {thread_count}')

plt.figure(figsize=(8, 4))
plt.imshow(binary, cmap='gray', aspect='auto',
           extent=[0, binary.shape[1], angles[-1], 0])
plt.xlabel('Смещение ρ (пиксели)')
plt.ylabel('Угол φ (градусы)')
plt.title(f'Шаг 5: Бинарная синограмма ({thread_count} нитей)')
plt.tight_layout()
plt.show()

## Результат — обратная проекция

Из задания:

> *«Требуется построить обратную проекцию данной бинарной синограммы»*

Каждая единица в бинарной синограмме = одна нить. Протягиваем все нити и смотрим, что получилось.
Для контраста применяем *«контрастирование с зашкаливанием небольшого числа пикселей с высокой яркостью»*.

In [ ]:
lines, weights = binary_sinogram_to_lines(binary, angles, probabilities)
size = image.shape[0]

canvas = draw_threads(lines, size, weights=weights)

# Контрастирование (quantile 0.995 — «зашкаливание» по заданию)
positive = canvas[canvas > 0]
limit = np.quantile(positive, 0.995) if positive.size > 0 else 1.0
preview = np.clip(canvas, 0, limit)
preview /= preview.max()

# Инверсия для отображения: нити -> тёмные (как на фото)
display = 1.0 - preview

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Исходное фото')
axes[0].axis('off')
axes[1].imshow(display, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Результат ({thread_count} нитей)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Белые нити на чёрном фоне

По заданию: *«будем строить ниткографическое изображение на черном фоне… нить одного цвета — белого»*.

Ниже — как выглядит реальная картина из ниток (без инверсии, нити белые).

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(preview, cmap='gray', vmin=0, vmax=1)
plt.title('Белые нити на чёрном фоне (как в реальности)')
plt.axis('off')
plt.tight_layout()
plt.show()